<a href="https://colab.research.google.com/github/juanpablor69/Proyecto_IA/blob/main/03_logistic_regression_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Proyecto - Inteligencia Artificial para las Ciencias y las Ingenierías**
## Tercer Notebook con uso de Modelos SVM y Logistic Regression

Autor: Juan Pablo Rendón Jimenez. \
Universidad de Antioquia

El **objetivo** de este notebook será aproximarnos a la solucion final usando distintos modelos junto a estrategias de preprocesado.




## Enlace con Kaggle

In [1]:
import os
os.environ['KAGGLE_CONFIG_DIR'] = '.'
!chmod 600 ./kaggle.json
!kaggle competitions download -c udea-ai-4-eng-20252-pruebas-saber-pro-colombia

  0% 0.00/29.9M [00:00<?, ?B/s]
100% 29.9M/29.9M [00:00<00:00, 1.14GB/s]


## Lectura e inspeccion de datos

In [2]:
!unzip udea*.zip > /dev/null
!wc *.csv

   296787    296787   4716673 submission_example.csv
   296787   4565553  59185238 test.csv
   692501  10666231 143732437 train.csv
  1286075  15528571 207634348 total


In [3]:
import pandas as pd
import numpy as np

datos = pd.read_csv("train.csv")
print("Dimensiones del dataset:", datos.shape)

Dimensiones del dataset: (692500, 21)


La base de datos contiene 692500 filas y 21 columnas. La estructura es la siguiente:

In [4]:
datos.head()

,ID,PERIODO_ACADEMICO,E_PRGM_ACADEMICO,E_PRGM_DEPARTAMENTO,E_VALORMATRICULAUNIVERSIDAD,E_HORASSEMANATRABAJA,F_ESTRATOVIVIENDA,F_TIENEINTERNET,F_EDUCACIONPADRE,F_TIENELAVADORA,...,E_PRIVADO_LIBERTAD,E_PAGOMATRICULAPROPIO,F_TIENECOMPUTADOR,F_TIENEINTERNET.1,F_EDUCACIONMADRE,RENDIMIENTO_GLOBAL,INDICADOR_1,INDICADOR_2,INDICADOR_3,INDICADOR_4
0,904256,20212,ENFERMERIA,BOGOTÁ,Entre 5.5 millones y menos de 7 millones,Menos de 10 horas,Estrato 3,Si,Técnica o tecnológica incompleta,Si,...,N,No,Si,Si,Postgrado,medio-alto,0.322,0.208,0.310,0.267
1,645256,20212,DERECHO,ATLANTICO,Entre 2.5 millones y menos de 4 millones,0,Estrato 3,No,Técnica o tecnológica completa,Si,...,N,No,Si,No,Técnica o tecnológica incompleta,bajo,0.311,0.215,0.292,0.264
2,308367,20203,MERCADEO Y PUBLICIDAD,BOGOTÁ,Entre 2.5 millones y menos de 4 millones,Más de 30 horas,Estrato 3,Si,Secundaria (Bachillerato) completa,Si,...,N,No,No,Si,Secundaria (Bachillerato) completa,bajo,0.297,0.214,0.305,0.264
3,470353,20195,ADMINISTRACION DE EMPRESAS,SANTANDER,Entre 4 millones y menos de 5.5 millones,0,Estrato 4,Si,No sabe,Si,...,N,No,Si,Si,Secundaria (Bachillerato) completa,alto,0.485,0.172,0.252,0.190
4,989032,20212,PSICOLOGIA,ANTIOQUIA,Entre 2.5 millones y menos de 4 millones,Entre 21 y 30 horas,Estrato 3,Si,Primaria completa,Si,...,N,No,Si,Si,Primaria completa,medio-bajo,0.316,0.232,0.285,0.294


## Analisis de la base de datos:

In [5]:
print(datos.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 692500 entries, 0 to 692499
Data columns (total 21 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   ID                           692500 non-null  int64  
 1   PERIODO_ACADEMICO            692500 non-null  int64  
 2   E_PRGM_ACADEMICO             692500 non-null  object 
 3   E_PRGM_DEPARTAMENTO          692500 non-null  object 
 4   E_VALORMATRICULAUNIVERSIDAD  686213 non-null  object 
 5   E_HORASSEMANATRABAJA         661643 non-null  object 
 6   F_ESTRATOVIVIENDA            660363 non-null  object 
 7   F_TIENEINTERNET              665871 non-null  object 
 8   F_EDUCACIONPADRE             669322 non-null  object 
 9   F_TIENELAVADORA              652727 non-null  object 
 10  F_TIENEAUTOMOVIL             648877 non-null  object 
 11  E_PRIVADO_LIBERTAD           692500 non-null  object 
 12  E_PAGOMATRICULAPROPIO        686002 non-null  object 
 13 

In [6]:
print("\n--- Valores faltantes por columna ---")
print(datos.isnull().sum())


--- Valores faltantes por columna ---
ID                                 0
PERIODO_ACADEMICO                  0
E_PRGM_ACADEMICO                   0
E_PRGM_DEPARTAMENTO                0
E_VALORMATRICULAUNIVERSIDAD     6287
E_HORASSEMANATRABAJA           30857
F_ESTRATOVIVIENDA              32137
F_TIENEINTERNET                26629
F_EDUCACIONPADRE               23178
F_TIENELAVADORA                39773
F_TIENEAUTOMOVIL               43623
E_PRIVADO_LIBERTAD                 0
E_PAGOMATRICULAPROPIO           6498
F_TIENECOMPUTADOR              38103
F_TIENEINTERNET.1              26629
F_EDUCACIONMADRE               23664
RENDIMIENTO_GLOBAL                 0
INDICADOR_1                        0
INDICADOR_2                        0
INDICADOR_3                        0
INDICADOR_4                        0
dtype: int64


## Limpieza de datos
Luego del analisis realizado identificamos dos columnas completamente identicas (Internet), por lo tanto procederemos con su eliminacion. Ademas, observamos un alto numero de valores faltantes, por lo que rellenaremos esos espacios con un NO REPORTA para evitar errores o sesgos.

In [7]:
# Elimino columna duplicada
if 'F_TIENEINTERNET.1' in datos.columns:
    datos.drop(columns=['F_TIENEINTERNET.1'], inplace=True)

# IMPUTACION DE DATOS FALTANTES
cols_categoricas = [
    'E_VALORMATRICULAUNIVERSIDAD', 'E_HORASSEMANATRABAJA',
    'F_ESTRATOVIVIENDA', 'F_TIENEINTERNET', 'F_EDUCACIONPADRE',
    'F_TIENELAVADORA', 'F_TIENEAUTOMOVIL', 'E_PAGOMATRICULAPROPIO',
    'F_TIENECOMPUTADOR', 'F_EDUCACIONMADRE'
]

for col in cols_categoricas:
    if col in datos.columns:
        datos[col] = datos[col].fillna('no info')

columnas_categoricas = [
    'E_VALORMATRICULAUNIVERSIDAD', 'E_HORASSEMANATRABAJA', 'F_ESTRATOVIVIENDA',
    'F_TIENEINTERNET', 'F_EDUCACIONPADRE', 'F_TIENELAVADORA',
    'F_TIENEAUTOMOVIL', 'F_TIENECOMPUTADOR', 'F_EDUCACIONMADRE'
]

for col in columnas_categoricas:
    datos[col] = datos[col].fillna('no info')

datos.fillna(0, inplace=True)

# === Mapeo ordinal: Valor matrícula ===
mapa_matricula = {
    'Menos de 500 mil': 0.25,
    'Entre 500 mil y menos de 1 millón': 0.75,
    'Entre 1 millón y menos de 2.5 millones': 1.75,
    'Entre 2.5 millones y menos de 4 millones': 3.25,
    'Entre 4 millones y menos de 5.5 millones': 4.75,
    'Entre 5.5 millones y menos de 7 millones': 6.25,
    'Más de 7 millones': 7.75,
    'No pagó matrícula': 0,
    'no info': -1
}
datos['E_VALORMATRICULAUNIVERSIDAD'] = datos['E_VALORMATRICULAUNIVERSIDAD'].map(mapa_matricula)

datos['F_EDUCACIONMADRE'] = datos['F_EDUCACIONMADRE'].replace(
    ['No sabe', 'No Aplica'], 'no info'
)



## Conversion de columnas en one-hot

En la entrega 3 usaremos modelos basados en Random Forest, entre otros. Para hacer uso de estos modelos necesitaremos convertir algunos datos ya que ellos solo trabajan con valores numéricos. En este caso vamos a reemplazar los datos como "Si" o "No" por una variable booleana como 0 y 1.

In [8]:
# === One-hot encoding automático ===
datos = pd.get_dummies(datos, columns=['F_EDUCACIONMADRE'], prefix='EDUMADRE')

# === Confirmar cambios ===
print("\n--- Limpieza completada ---")
print(datos.info())


--- Limpieza completada ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 692500 entries, 0 to 692499
Data columns (total 30 columns):
 #   Column                                         Non-Null Count   Dtype  
---  ------                                         --------------   -----  
 0   ID                                             692500 non-null  int64  
 1   PERIODO_ACADEMICO                              692500 non-null  int64  
 2   E_PRGM_ACADEMICO                               692500 non-null  object 
 3   E_PRGM_DEPARTAMENTO                            692500 non-null  object 
 4   E_VALORMATRICULAUNIVERSIDAD                    692500 non-null  float64
 5   E_HORASSEMANATRABAJA                           692500 non-null  object 
 6   F_ESTRATOVIVIENDA                              692500 non-null  object 
 7   F_TIENEINTERNET                                692500 non-null  object 
 8   F_EDUCACIONPADRE                               692500 non-null  object 
 9   F_TIENEL

Como vemos en la tabla, tenemos toda la base de datos sin incongruencias ni valores faltantes.

In [9]:
datos.head()

,ID,PERIODO_ACADEMICO,E_PRGM_ACADEMICO,E_PRGM_DEPARTAMENTO,E_VALORMATRICULAUNIVERSIDAD,E_HORASSEMANATRABAJA,F_ESTRATOVIVIENDA,F_TIENEINTERNET,F_EDUCACIONPADRE,F_TIENELAVADORA,...,EDUMADRE_Educación profesional incompleta,EDUMADRE_Ninguno,EDUMADRE_Postgrado,EDUMADRE_Primaria completa,EDUMADRE_Primaria incompleta,EDUMADRE_Secundaria (Bachillerato) completa,EDUMADRE_Secundaria (Bachillerato) incompleta,EDUMADRE_Técnica o tecnológica completa,EDUMADRE_Técnica o tecnológica incompleta,EDUMADRE_no info
0,904256,20212,ENFERMERIA,BOGOTÁ,6.25,Menos de 10 horas,Estrato 3,Si,Técnica o tecnológica incompleta,Si,...,False,False,True,False,False,False,False,False,False,False
1,645256,20212,DERECHO,ATLANTICO,3.25,0,Estrato 3,No,Técnica o tecnológica completa,Si,...,False,False,False,False,False,False,False,False,True,False
2,308367,20203,MERCADEO Y PUBLICIDAD,BOGOTÁ,3.25,Más de 30 horas,Estrato 3,Si,Secundaria (Bachillerato) completa,Si,...,False,False,False,False,False,True,False,False,False,False
3,470353,20195,ADMINISTRACION DE EMPRESAS,SANTANDER,4.75,0,Estrato 4,Si,No sabe,Si,...,False,False,False,False,False,True,False,False,False,False
4,989032,20212,PSICOLOGIA,ANTIOQUIA,3.25,Entre 21 y 30 horas,Estrato 3,Si,Primaria completa,Si,...,False,False,False,True,False,False,False,False,False,False


## Modelo Baseline con Regresión Logística y Preprocesamiento Clásico
Primero se realiza un preprocesamiento completo que incluye imputación de valores faltantes, normalización de variables numéricas y codificación One-Hot para las variables categóricas. Posteriormente, se entrena un modelo de Regresión Logística multiclase, elegido por su rapidez de entrenamiento y su comportamiento estable incluso en datasets con alta dimensionalidad.

### Entrenamiento del Modelo

In [10]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score

X = datos.drop(columns=["RENDIMIENTO_GLOBAL"])
y = datos["RENDIMIENTO_GLOBAL"]

# Crear el mapping automáticamente desde los valores únicos del target
clases = sorted(datos['RENDIMIENTO_GLOBAL'].unique())

rmap = {clase: idx for idx, clase in enumerate(clases)}
print("rmap generado automáticamente:")
print(rmap)

datos['RENDIMIENTO_GLOBAL'] = datos['RENDIMIENTO_GLOBAL'].map(rmap)

#separamos columnas
num_cols = X.select_dtypes(include=["int64","float64"]).columns
cat_cols = X.select_dtypes(include=["object"]).columns

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), num_cols),

        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_cols)
    ]
)

model = Pipeline([
    ("preprocess", preprocess),
    ("clf", LogisticRegression(max_iter=200, n_jobs=-1))
])

#Entrenamiento
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


model.fit(X_train, y_train)

#evaluacion
preds = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, preds))




rmap generado automáticamente:
{'alto': 0, 'bajo': 1, 'medio-alto': 2, 'medio-bajo': 3}
Accuracy: 0.4166642599277978


### Conclusion
Siguendo las indicaciones del profesor, hice un primer intento usando SMV pero se llevó mucho tiempo de ejecucion (mas de 15 minutos). A la final conclui que el dataset es muy grande para la complejidad de los modelos SMV.

Luego busque mas modelos que puedan servirme y encontré Logistic Regression. Se evaluo el desempeño del modelo sobre un conjunto de validación, obteniendo un accuracy aproximado del 0.416, lo cual sirve como línea base y evidencia la necesidad de modelos más complejos para mejorar el rendimiento.

In [11]:
# ================================
#   PREDICCIONES SOBRE EL TEST
# ================================

# 1. Cargar el test
test_data = pd.read_csv("test.csv")

# Guardar IDs antes de modificar
test_ids = test_data["ID"]


# 3. Imputar categóricas igual que en train
for col in cat_cols:
    if col in test_data.columns:
        test_data[col] = test_data[col].fillna('no info')

# 4. Imputar numéricas igual que en train
test_num_cols = test_data.select_dtypes(include=['int64','float64']).columns
test_data[test_num_cols] = test_data[test_num_cols].fillna(0)

# 5. Aplicar transformación ordinal a matrícula (igual al train)
mapa_matricula = {
    'Menos de 500 mil': 0.25,
    'Entre 500 mil y menos de 1 millón': 0.75,
    'Entre 1 millón y menos de 2.5 millones': 1.75,
    'Entre 2.5 millones y menos de 4 millones': 3.25,
    'Entre 4 millones y menos de 5.5 millones': 4.75,
    'Entre 5.5 millones y menos de 7 millones': 6.25,
    'Más de 7 millones': 7.75,
    'No pagó matrícula': 0,
    'no info': -1
}
test_data['E_VALORMATRICULAUNIVERSIDAD'] = test_data['E_VALORMATRICULAUNIVERSIDAD'].map(mapa_matricula)

# 6. Reemplazar valores de educación madre igual que en train
test_data['F_EDUCACIONMADRE'] = test_data['F_EDUCACIONMADRE'].replace(
    ['No sabe', 'No Aplica'], 'no info'
)

# 7. Listo para predecir: NO eliminamos ID
X_test_final = test_data

# 8. Predicciones (en texto)
preds_test_data = model.predict(X_test_final)

# 9. No necesitamos rmapi porque ya están en texto
text_preds_test_data = preds_test_data

# 10. Crear submission
submission = pd.DataFrame({
    "ID": test_data["ID"],
    "RENDIMIENTO_GLOBAL": text_preds_test_data
})

# 11. Guardar CSV final
submission.to_csv("03_submission.csv", index=False)

submission.head()




,ID,RENDIMIENTO_GLOBAL
0,550236,bajo
1,98545,bajo
2,499179,alto
3,782980,bajo
4,785185,bajo


### Envio de Intento a Kaggle

In [13]:
!kaggle competitions submit -c udea-ai-4-eng-20252-pruebas-saber-pro-colombia -f 03_submission.csv -m "Juan Rendon notebook 03"

100% 3.96M/3.96M [00:00<00:00, 17.5MB/s]
Successfully submitted to UDEA/ai4eng 20252 - Pruebas Saber Pro Colombia